# First Principles

- `Data preprocessing: tokenizer`. From utf-8 bytes encoding to an entropy optimized encoding/decoding compression (5:1 ratio for SOTA tokenizers with around 100k vocab size)

- `Hypothesis Class: the architecture (Representation Capabilities)`

- `Loss (/Cost): how far are we from the real distribution?` KL distance -> Cross-entropy (differenciability & continuous gradient). Root Node of backward pass in autograd computation graph. Approximation of real cost through averaged batch sampling from data (supposed to represent the real distribution) (averaged over B & T for y/x sequence prediction)

- `Training: getting closer to reality. Naviguating through the search space.` Gradient averaged in batched (high batch size -> lower variance of search space representation): gives us direction. Backward pass from autograd computation graph populates gradients through learnable weights (needs activations memory storage). Optimizing module: gives us how to move in that direction

- `The training system (environment)` Doing the data pipeline flywheel from disk loading (where the dataset file lives) to sample processing with CPU and memory living in system RAM. data process method defined in Dataset Custom object, [idx] (.__get_item__() built-in method) calls this method

# Pre-processing: Tokenizer (.encode(str), .decode(np.array), .encode_file(in_str, out_str))

In [1]:
import fast_tokenizer

#tok = fast_tokenizer.FastTokenizer.train(files="../data/TinyStoriesV2-GPT4-valid.txt", out_dir="../data/my_tokenizer", vocab_size=4096, min_frequency=2, sep_token="<|endoftext|>")

tok = fast_tokenizer.FastTokenizer(tokenizer_json="../data/my_tokenizer/tokenizer.json", sep_token="<|endoftext|>")

#tok.encode_file(in_path="../data/TinyStoriesV2-GPT4-valid.txt", out_path="../data/TinyStoriesV2-GPT4-valid.npy")

In [2]:
import numpy as np

x = np.load('../data/TinyStoriesV2-GPT4-valid.npy', mmap_mode='r')
print(x[:100])
print(tok.decode(x[:100]))

[  85  867  499  510  266  328  614  373  263  914  471   12  343 2026
 2635  351 1568  284 3383  543  404  892  330  263  396  481   14  344
  283  385  802  267  263 3383 1084  552  266 2564  337   14  318 3380
  264 4087  337  267  398 1104  485  892   14  284 3383  366  512  341
  691  359   14  199    0  199  434  444  259  402   12  316  259 1064
  267 1155  947   12  407  283  259  347 3667   14  443  396  501  408
  423  534  266  325  827  263 3667   14  536  329   12  423 1063  341
  675  476]
u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball


# Transformer LM Modules from scratch

In [3]:
import torch
from einops import rearrange, einsum, reduce, repeat
from torch import nn
import math

### Linear Layer

In [4]:
# ALl neural nets modules should inherent from nn.Module parent class -> inherits convenient methods such as: load_state_dict(), to(), get_parameters(), cpu(), cuda(), children(), bfloat16()...

# Implement a Linear Class (= "a Linear Module")
# y = xWT

class Linear(nn.Module): # Inherits nn.Module methods()
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        
        self.sigma = (2 / (in_features + out_features)) ** (1/2)

        self.weight = nn.Parameter(nn.init.trunc_normal_(torch.empty(out_features, in_features, dtype=dtype, device=device), 
                                                                mean=0, std = self.sigma, 
                                                                a = -3 * self.sigma, b= 3 * self.sigma ))

    def forward(self, x: torch.tensor) -> torch.Tensor: # All nn.Module need to have a forward() method
        return einsum(x, self.weight, '... in_feature , out_feature in_feature -> ... out_feature')

### Embedding Look-up Table

In [5]:
# Create an embedding table Class
# Ounce again, every neural nets module should inherent nn.Module for convenient access to parent methods (.load_state_dict(), .Parameters(), .to()...)

class Embedding(nn.Module):

    def __init__(self, num_embeddings, embedding_dim, dtype=None, device=None):
        super().__init__()
        self.weight = nn.Parameter(nn.init.trunc_normal_(torch.empty(num_embeddings, embedding_dim, dtype=dtype, device=device), std=1, a=-3, b=3))

    def forward(self, x: torch.LongTensor) -> torch.Tensor: # (... T) -> (... T d_model)
        return self.weight[x] # shape: = x.shape + self.weights.shape[1:]

### Layer Norm: RMSNorm

In [6]:
# Implement LayerNorm: RMSNorm
# Dtype: Prevent overflow of root mean square by using dtype = float32. 
# It is possible to change data type in this way: input_dtype -> float32 -> input_dtype

class RMSNorm(nn.Module):

    def __init__(self, d_model, eps=1e-5, dtype=None, device=None):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device)) # (d_model)
        self.eps = eps

    def forward(self, x): # (b T d_model -> b T d_model)
        in_dtype = x.dtype
        x = x.to(dtype = torch.float32)
        batched_rms = torch.sqrt(reduce(torch.square(x), '... d_model -> ... 1', reduction = 'mean') + self.eps)
        x_norm = torch.div(x, batched_rms)
        result = einsum(x_norm, self.weight.to(dtype=torch.float32), '... d_model, d_model -> ... d_model')
        return result.to(dtype=in_dtype)

### FFN: SwiGLU

In [7]:
# FFN Transformer Layer
# Swish activation function + Gated Linear Unit: W1 & W2, thus d_ffn becomes 2/3 what it would be without gated linear unit to keep same parameter count
# d_model -> d_ffn -> d_model. One single hidden layer. d_ffn = 8/3 * d_model (should be a multiplier of 64 for hardware efficiency)
# The module should - as always - inherent from PyTorch nn.Module parent class for convenient methods usage (.to(), .load_state_dict(), .parameters())
# Weights should be defined inside nn.Parameter() inside the child module for compatibility with Torch other parameter related methods (e.g. .parameters())

class SwiGLU_FFN(nn.Module):
    
    def __init__(self, d_model, d_ff=None, dtype=None, device=None):
        super().__init__()
        self.d_ff = d_ff
        if not self.d_ff:
            self.d_ff = int(((8/3) * d_model // 64) * 64)  # Kepping same parameter count with/without Gated Linear Unit
        self.w1 = Linear(in_features=d_model, out_features=self.d_ff, dtype=dtype, device=device)
        self.w2 = Linear(in_features=self.d_ff, out_features=d_model, dtype=dtype, device=device)
        self.w3 = Linear(in_features=d_model, out_features=self.d_ff, dtype=dtype, device=device)

    def forward(self, x): # x.shape == ([B T d_model])
        x_1 = self.w1(x) # ... d_model -> ... d_ff
        GLU = self.w3(x)
        SiLU = einsum(x_1, torch.sigmoid(x_1), '... d_ff, ... d_ff -> ... d_ff')
        SwiGLU = einsum(SiLU, GLU, '... d_ff, ... d_ff -> ... d_ff')
        return self.w2(SwiGLU)

### Positional Embedding: RoPE

In [8]:
# Relative Positional Embedding (RoPE)
# From (nn.Module). Uses buffer for the rotation angles with 'self.register_buffer()'
# Different dimensions of the latent space get different rotation speed. Rotation itself depends on the position of the token in the sequence.
# In the Latent Space, delta of angle between vectors is linearly proportional to the distance of their index in the sequence

class RoPE(nn.Module):
    # Create a ([T, d_k, d_k]) tensor of the rotation matrices is suboptimal
    # Create a ([T, d_k/2, 2, 2]) tensor of rotation matrices, rearrange x to become (... d_k/2 2) then MatMul then back to (... d_k)
    def __init__(self, theta: int, d_k: int, max_sequence_len: int, device=None, dtype=None):
        super().__init__()
        self.thetas_dim = theta ** (-torch.arange(0, d_k, step=2, device=device, dtype=dtype) / d_k)
        self.thetas_sequence = einsum(torch.arange(0, max_sequence_len, device=device, dtype=dtype), self.thetas_dim, 'maxT, d2 -> maxT d2')
        self.stack = torch.stack([torch.cos(self.thetas_sequence), -torch.sin(self.thetas_sequence), 
                                torch.sin(self.thetas_sequence), torch.cos(self.thetas_sequence)]) # (4, maxT, d_k/2)
        self.register_buffer('RoPE', rearrange(self.stack, ' (l c) maxT d2 -> maxT d2 l c', l=2, c=2), persistent=False)
    
    def forward(self, x, token_positions): # x size: (... T d_k) / token_positions size: (... T)
        sequence_rope = self.RoPE[token_positions] # (... T d2 l c)
        x_paired = rearrange(x, '... T (d1 d2) -> ... T d1 d2', d2=2)
        output_paired = einsum(sequence_rope, x_paired, '... T dk2 l c, ... T dk2 c -> ... T dk2 l')
        return rearrange(output_paired, '... T dk2 l -> ... T (dk2 l)')

### Softmax: enables normalization in an expressive matter. Creates attention scores & probabilities from logits

In [9]:
def Softmax(x: torch.Tensor, dim: int):   
    x_copy = torch.transpose(x, dim, -1) # (... i)
    x_minus_max = reduce(x_copy, '... i -> ... 1', reduction='max')
    x_copy = x_copy - x_minus_max # Stability trick to avoidd exp(vi) to become inf and then having inf/inf = NaN
    x_copy = torch.exp(x_copy)
    x_div = reduce(x_copy, '... i -> ... 1', reduction='sum')
    probs = torch.div(x_copy, x_div)
    return torch.transpose(probs, dim, -1)

### Attention Layer: enables information to flow in a causal matter

In [10]:
def scaled_dot_product_attention(Q, K, V, mask=True):
    T1 = Q.shape[-2]
    T2 = K.shape[-2]
    true_mask = torch.ones(T1,T2, dtype=torch.bool, device=mask.device)
    mask_copy = mask * true_mask
    mask_matrix = torch.zeros_like(mask_copy, dtype=torch.float, device=mask.device)
    mask_matrix[~mask_copy] = float('-inf')
    d_k = K.shape[-1]
    logits_scores = torch.div(einsum(Q, K, '... T1 d_k, ... T2 d_k -> ... T1 T2'), ( d_k ** (1/2) ))
    mask_matrix = mask_matrix.to(device=Q.device)
    logits_scores += mask_matrix
    scores = Softmax(logits_scores, dim=-1)
    return einsum(scores, V, '... T1 T2, ... T2 d_v -> ... T1 d_v')

In [11]:
# Causal Multi Head Self Attention Module inherits from parent torch class nn.Module for methods like state_dict(), buffers(), to()...
# Three Linear Layer Matrices of learnable weights (no biases): WQ, WK, WV
# max_sequence_length is NOT necessarily input sequence length. T is the input size but it can be truncated and not be of the full context size of the LM!
# token_positions should be torch.arange(x.shape[1], device=x.device)

class causal_multihead_self_attention_with_rope(nn.Module):

    def __init__(self, d_model: int, num_heads: int, max_sequence_length:int, rope_theta=10000, device=None, dtype=None):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divided by num_heads'
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.q_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.k_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.v_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.output_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.rope = RoPE(theta=rope_theta, d_k=self.d_k, max_sequence_len=max_sequence_length, device=device)
        mask = torch.tril(torch.ones(max_sequence_length, max_sequence_length, dtype=torch.bool, device=device))
        self.register_buffer('mask', mask, persistent=False)
    
    def forward(self, x, token_positions=None): # (B T d_model)
        if token_positions is None:
            token_positions = torch.arange(x.shape[1], device=x.device)
        T = x.shape[1]
        Q, K, V = rearrange(self.q_proj(x), '... T (nh dk) -> ... nh T dk', nh=self.num_heads), rearrange(self.k_proj(x), '... T (nh dk) -> ... nh T dk', nh=self.num_heads), rearrange(self.v_proj(x), '... T (nh dk) -> ... nh T dk', nh=self.num_heads) # (B num_h T d_k)
        Q, K = self.rope(Q, token_positions=token_positions), self.rope(K, token_positions=token_positions)
        output = scaled_dot_product_attention(Q, K, V, mask=self.mask[:T,:T]) # (B n_h T d_k)
        return self.output_proj(rearrange(output, '... nh T dk -> ... T (nh dk)', nh=self.num_heads))

### Transformer Module: general representation capabilities & harware efficient (from heavy tensor contractions)

In [12]:
# Transformer Block Module

class Transformer_Block(nn.Module):

    def __init__(self, d_model:int, d_ff:int, num_heads:int, max_sequence_len:int, theta=10000, device=None, dtype=None):
        super().__init__()
        self.ln1 = RMSNorm(d_model=d_model, dtype=dtype, device=device)
        self.attn = causal_multihead_self_attention_with_rope(d_model=d_model, num_heads=num_heads, max_sequence_length=max_sequence_len, rope_theta=theta, device=device, dtype=dtype) # We need max_seq_len for RoPE buffer init when instantiating the Transformer_Block Module
        self.ln2 = RMSNorm(d_model=d_model, dtype=dtype, device=device)
        self.ffn = SwiGLU_FFN(d_model=d_model, d_ff=d_ff, dtype=dtype, device=device)
    
    def forward(self, x): # (B T d_model -> B T d_model)
        y = x + self.attn(self.ln1(x))
        output = y + self.ffn(self.ln2(y))
        return output

In [13]:
# Transformer LM Module
# 3 keys for a good hypothesis class: representation capability + hardware efficient + stable learning process
# transformer_lm facts:
# context_len is the max_context_len: x can actually be processed even if T (sequence_len) < context_len
# num_blocks is num_layers a.k.a the length ratio
# d_model is the width ratio
# wight / length ratio of language models should be around 100 (from open source ablation)
# For hardware efficiency reasons all matrices width and len should be multiples of 32. GPU thread block contains 32 threads that apply the same operation synchronously on part of the shared memory of the block) (warp?)
# num_heads * d_k should be equal to d_model with the constraint that d_k and d_model must be multiples of 32 (Hardware efficiency)

class transformer_lm(nn.Module):

    def __init__(self, vocab_size:int, d_model:int, d_ff:int, num_heads:int, num_blocks:int, context_length:int, theta=10000, device=None, dtype=None):
        super().__init__()
        self.token_embeddings = Embedding(num_embeddings=vocab_size, embedding_dim=d_model, dtype=dtype, device=device)
        self.layers = nn.Sequential(*[Transformer_Block(d_model=d_model, d_ff=d_ff, num_heads=num_heads, max_sequence_len=context_length, theta=theta, device=device, dtype=dtype) for _ in range(num_blocks)])
        self.ln_final = RMSNorm(d_model=d_model, dtype=dtype, device=device)
        self.lm_head = Linear(in_features=d_model, out_features=vocab_size, dtype=dtype, device=device)
    
    def forward(self, x): # (B T -> B T vocab_size)
        x_embd = self.token_embeddings(x) # (B T d_model)
        for i in range(len(self.layers)):
            x_embd = self.layers[i](x_embd)
        x_embd = self.ln_final(x_embd)
        return self.lm_head(x_embd)

### Config file

In [14]:
# Dataclass config file (use .asdict() for arguments input)
# Serialize (via json) and save to disk
# Also can be a cfg.py python file that gets imported with import cfg
# type hint for @dataclass __init__() helper built-in function to work at instanciation

from dataclasses import dataclass
import json

@dataclass
class Config:
    vocab_size: int = 1000
    d_model: int = 64
    d_ff: int = 256
    num_heads: int = 4
    num_blocks: int = 3
    context_length: int = 10
    device: str = "mps"

# Import cfg
# cfg = cfg.Config()
# LM = transformer_lm(**cfg.__dict__)

cfg = Config()
print(cfg.__dict__)

#with open('../data/config.json', 'w') as f:
#    json.dump(cfg.__dict__, f, indent=1)

{'vocab_size': 1000, 'd_model': 64, 'd_ff': 256, 'num_heads': 4, 'num_blocks': 3, 'context_length': 10, 'device': 'mps'}


# Training FLOPs and Memory accounting

## TL;DR
- `FLOPs(transformer_lm) of one forward pass per sequence = num_layers * (24 * n * d_model^2 + 4 * d_model * n^2) + 2 * n * d_model * vocab_size`
- `Total Paramters = num_layers * (12 * d_model^2 + 2 * d_model) + d_model * (2*vocab_size + 1)`
- `Memory capacity required for training = batch_size * context_length * (2 - recomputation_prop) * activation count + 16 * parameter_count = activations + Model State = batch_size * a + b`
- `Total training time of GPT2-XL on one single H100 = 17.2e6 s = 4,777 hours = 200 days`

For `transformer_lm(**config.__dict__)` that would have this specific config

- **Config: (GPT2-XL)**
  - `vocab_size: int = 50,257`
  - `context_length: int = 1,024`
  - `num_layers: int = 48`
  - `d_model: int = 1,600`
  - `num_heads: int = 25`
  - `d_ff: int = 4,288` (the nearest multiple of 64 to 8/3 × 1,600)

- **Flops Accounting**
  - FLOPS(Self_MHA) = 3 * 2 (for: * & +) * n * dmodel^2 (for: Q, K, V) + 2 * h * d_k * n^2 (for: Q, Kt) + 2 * h * d_k * n^2 (for: attn V) + 2 * n * d_model^2 (for: V Output_proj)
  - FLOPs(Self MHA) = 8nd^2 + 4dn^2
  - (note: FLOPs(Softmax) = 3 * h * n^2 so o(FLOPs of the rest of attention module forward pass))
  - (note: FLOPs(RoPE) = O(n * d_model) so o(FLOPs of the rest of attention module forward pass))

  - FLOPs(SwiGLU FFN) = 3 * 2 * 8/3 * n * d_model^2 (3 MatMul cause of GLU, all have same FLOPs)
  - FLOPs(SwiGLU FFN) = 16 * nd^2
  - (note: d_ff = 8/3 * d_model)

- `FLOPs(transformer_lm) = num_layers * (24 * n * d_model^2 + 4 * d_model * n^2) + 2 * n * d_model * vocab_size`
- (note: embedding table is lookup so complexity in O(context_length) & RMSNorm is O(n*d_model) so dropped too)

- **Parameter Accounting**
  - Embedding Layer = vocab_size * d_model
  - MHA = 4 * d_model^2`
  - FFN = 8 * d_model^2`
  - RMSNorm = d_model
  - Tranformer Block = num_layers * (12 * d_model^2 + 2 * d_model)
  - Final Linear Projection = d_model * vocab_size
- `Total Paramters = num_layers * (12 * d_model^2 + 2 * d_model) + d_model * (2*vocab_size + 1)`

 ![image.png](../images/image.png)

## HBM Memory Capacity during training
- During training there is a tradeoff between Memory Capacity and recomputation (flash-attention principle)
- We need/store all activations values for .backward() gradient propagation through the weights. We can store all of them in memory or recompute some portion of it during backpropagation (flash-attention). 
- Fusing kernels in backprop helps going faster by trading reducing HBM-SharedMemory read/write accesses bottlenecks with increased memory capacity.
- (note: a kernel is a low-level function that gives instruction to a GPU. Kernels can be written for micro operations like +/* or macro operations like softmax/attention/backprop for hardware optimization)
- Flash-attention uses fused kernels + re-computation for the three attention operations (QK + Softmax + AttnV) to handle memory capacity bottleneck. Softmax stats are saved to enable fast recomputation and no memory capacity (QK rows max & sum). Q,K,V are also saved in memory.

- Mixed-precision with bf16 for the weights, gradients and activations & fp32 for moments of order 1 and 2 in AdamW
  - Memory capacity required for training = `Model State` + `Activations`
  - `Memory capacity required for training = (2 + 2 + 4 + 4 + 4) * parameter_count + batch_size * (2 - recomputation_prop) * activation count (for the full sequence)`
  - (note: activation count scales quadratically with the context length: QK + softmax(QK) + AttnV = 3 * h * n^2 activations to store in memory. Flash-attention reduces this by recomputing activations during backward pass)
  - (note: AdamW stores a fp32 representation copy of weights during updates then cast weight back to bf16 for next forward pass)

![image.png](../images/training_memory.png)

![image.png](../images/memory_vs_context.png)

## Training Total FLOPs Accounting (for GPT2-XL)

- `AdamW update step = 15-20 * parameter_count FLOPs = 30 GigaFLOPs (negligable compared to forward and backward passes)`
- `One forward pass = 3.5 TFLOPs / sequence`
- `One backward pass = 7 TFLOPs / sequence (around two times more than the forward pass)`
- One step of batch_size 1024 = 10.5 * 1024 TeraFLOPs = 10.7e3 TeraFLOPs
- `400k steps training = 4.3e9 TeraFLOPs`
- H100 theorical peak for float32 = 495 TeraFLOPs / s
- Assume MFU = 0.5
- `Total training time of GPT2-XL on one single H100 = 17.2e6 s = 4,777 hours = 200 days`

# Loss (/cost) & Learning (grads + optimizer) & Stability tricks

### Cross-entropy Implementation

In [15]:
# Cross-entropy loss. Scalar value of the current cost. Approximated through batched sampling. Root node of backpropagaton pass -> non-convex optimization

def cross_entropy_loss(logits, targets) -> torch.float: # (... Vocab_size), (...)
    max_logit = reduce(logits, '... V -> ... 1', reduction='max')
    exp_norm_logits = torch.exp(logits - max_logit)
    sum_exps = reduce(exp_norm_logits, '... V -> ... 1', reduction='sum')
    gathered_logits = torch.gather(logits, dim=-1, index=repeat(targets, '... -> ... c', c=1)) # (...)
    log_probs = gathered_logits - max_logit - torch.log(sum_exps)
    return - ( reduce(log_probs, '... -> ', reduction='mean')) # averaged cost over batch and sequence

def cross_entropy_loss_logsumexp_fused_kernel(logits, targets): # 3 main kernels instead of 6: log + sum + exp + max get fused into ONE single kernel. Less HMB read/write -> higher arithmetic intensity operation
    gathered_logits = torch.gather(input=logits, dim=-1, index=targets.unsqueeze(-1) )
    batched_cross_entropy = - gathered_logits + torch.logsumexp(logits, dim=-1, keepdim=True) # (B T 1)
    return reduce(batched_cross_entropy, '... -> ', reduction='mean')

### SGD and AdamW Implementations

In [16]:
# Learning
# SGD with learning rate decreasing over time from Pytorch API

from collections.abc import Callable, Iterable
from typing import Optional

class SGD(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):

        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")

        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or 0.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.

        return loss


weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=100)

for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.item())
    loss.backward()
    opt.step()


31.556297302246094
31.556291580200195
5.414204120635986
0.12957409024238586
2.2718860954392707e-16
2.5321578465218153e-18
8.5266669346064e-20
5.079394235951715e-21
4.357432370564496e-22
4.84159152284944e-23


In [17]:
# Implement AdamW: (lr, weight_decay, beta1, beta2)
# Adam optimizer tracks a running the gradients moment of order 1 (hyperparameter: beta 1) and order 2 (beta 2)
# AdamW adds a weight decay at the optimizer .step() (instead of at the gradient level for example)

class AdamW(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3, weight_decay=0.01, betas=(0.9, 0.95), eps=1e-8):

        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")

        defaults = {"lr": lr, 'weight_decay': weight_decay, 'betas': betas, 'eps': eps}
        super().__init__(params, defaults)

    def step(self, closure = None):
        loss = None if closure is None else closure()

        for group in self.param_groups:
            lr, weight_decay, (beta1, beta2), eps = group["lr"], group["weight_decay"], group["betas"], group["eps"] # Get the hyperparameters
            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p] # Get state associated with p
                if len(state) == 0:
                    state['moment_order_1'] = torch.zeros_like(p)
                    state['moment_order_2'] = torch.zeros_like(p)
                    state['t'] = 1

                t = state['t']
                grad = p.grad
                moment_order_1 = state['moment_order_1'] * beta1 + (1 - beta1) * grad
                moment_order_2 = state['moment_order_2'] * beta2 + (1 - beta2) * grad**2
                adjusted_lr = lr * math.sqrt(1 - beta2**t) / (1 - beta1**t)
                with torch.no_grad():
                    p -= p * weight_decay * lr + adjusted_lr * moment_order_1 / (torch.sqrt(moment_order_2) + eps) # AdamW weight update. -= makes the update in_place, very important!

                state['moment_order_1'] = moment_order_1 # stateful buffer of total size 4 * num_param (torch.float32)
                state['moment_order_2'] = moment_order_2 # stateful buffer of total size 4 * num_param (torch.float32)
                state["t"] = t + 1 

        return loss

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = AdamW([weights], lr=100)

for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.item())
    loss.backward()
    opt.step()

25.811655044555664
9999.96484375
5002.7578125
68.2589340209961
8.469625473022461
1.3813329935073853
0.26083651185035706
0.0536811463534832
0.011623675003647804
0.00258781760931015


### Learning Rate Scheduling & Gradient Clipping (for stabilizing training)

In [18]:
# Learning Rate Scheduling 

def cosine_lr_schedule(t: int, lr_min: float, lr_max: float, T_warmup: int, T_c: int) -> float:
    '''
    Returns the learning rate at time t from the cosine_annealing scheduler
    '''
    if t < T_warmup:
        return t / T_warmup * lr_max
    if t <= T_c:
        return lr_min + 1/2 * (1 + math.cos((t-T_warmup) / (T_c - T_warmup) * math.pi)) * (lr_max - lr_min)
    else:
        return lr_min

In [19]:
# Gradient Clipping Implementation

def gradient_clipping(parameters_list: list[torch.tensor], max_grad: torch.float):
    '''
    Modifies parameters gradients in_place to cap the global l2-norm at max_grad
    '''
    running_l2_grad = 0

    for p in parameters_list:

        if p.grad is None:
            continue

        running_l2_grad += reduce(torch.square(p.grad), '... -> ', reduction='sum')
    
    l2_grad = torch.sqrt(running_l2_grad)

    if l2_grad > max_grad:
        for p in parameters_list:
            if p.grad is not None:
                p.grad *= max_grad / (l2_grad + 1e-6)

# Training System

## Data loader & checkpointing

In [20]:
# System training environment: load from disk (dataset) to system RAM + dataprocessing (CPU work) to GPU HBM (PCIe connection from system RAM to HBM)
# List of tensors (from numpy) into torch.stack

dataset = np.load('../data/TinyStoriesV2-GPT4-valid.npy', mmap_mode='r')

def get_batch(dataset, batch_size, context_len, device=None):
    batch_ids = np.random.randint(low=0, high=len(dataset) - context_len, size=(batch_size,))
    x_batch = []
    y_batch = []
    for i in batch_ids:
        x = dataset[i:i+context_len]
        y = dataset[i+1:i+context_len+1]
        x = np.copy(x)
        y = np.copy(y)
        x_batch.append(torch.from_numpy(x))
        y_batch.append(torch.from_numpy(y))
    x_batch, y_batch = torch.stack(x_batch), torch.stack(y_batch)
    x_batch, y_batch = x_batch.to(dtype=torch.int64), y_batch.to(dtype=torch.int64)
    return x_batch.to(device), y_batch.to(device)

In [21]:
# Save checkpoint states to disk for re-usability. Save and load the checkpoint dict with keys 'model_state', 'optimizer_state' and 't'. States are dict

def save_checkpoint(model, optimizer, iteration, out):
    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 't': iteration}, out)

def load_checkpoint(path, model, optimizer):

    checkpoint = torch.load(path)

    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    return checkpoint['t']